# SPK-1 ⭐ — Data / Partition Skew

**Break → Detect → Fix → Prove.** One key holds 90% of the rows, so one task does almost all
the work while the rest sit idle — the most common Spark performance failure.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch it while the cells run.

**Laptop-safe:** the data is *generated lazily* and only `count()`-ed — never collected or
written — so nothing fills memory or disk. Skew is a task-time problem, not a memory problem, so
the default **tuned** box is fine (no need for `make up-constrained`). Nothing to delete at the end.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import skewed_keys, key_dimension
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run — skew shows up in Stages -> Tasks -> Summary Metrics.
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — Parameters & the skewed dataset

We synthesize an `orders` fact skewed by `customer_id` (one "house account", `customer_id = 0`,
holds 90% of orders) and a small `customers` dimension — all lazily, nothing stored.

In [ ]:
# Lower N_ROWS to 5-10M if your laptop is slow; raise it to make the straggler more dramatic.
N_ROWS       = 20_000_000
N_COLD_KEYS  = 2_000      # "normal" customers the other 10% of orders spread across
HOT_KEY      = 0          # the marketplace "house account"
HOT_FRACTION = 0.90

fact = skewed_keys(spark, n_rows=N_ROWS, hot_key_fraction=HOT_FRACTION,
                   n_cold_keys=N_COLD_KEYS, hot_key=HOT_KEY, key_col="customer_id")
dim  = key_dimension(spark, n_keys=N_COLD_KEYS + 1, key_col="customer_id")

print(f"fact: ~{N_ROWS:,} orders (lazy)    dim: {N_COLD_KEYS + 1:,} customers")

In [ ]:
# Confirm the skew — the hot key dominates (small result, safe to show):
(fact.groupBy("customer_id").count()
     .orderBy(F.desc("count"))
     .limit(5)
     .show())

## 1. Break it — skewed sort-merge join

The `constrained` session profile turns **AQE off** (no runtime skew rescue) and **broadcast off**
(`autoBroadcastJoinThreshold = -1`), forcing a **sort-merge join** that shuffles both sides by
`customer_id`. Every hot-key row lands in the same reduce partition -> one straggler task.

> **Why aggregate `amount` instead of a bare `.count()`?** AQE skew-join detects a skewed partition by its **size in bytes**. A `.count()` lets Catalyst prune the join to the (highly compressible) key column, so the hot partition's shuffle shrinks to a few KB and AQE can't see the skew — even though one task still does all the row work. Carrying a real payload (`sum(amount)`, matching the revenue scenario) gives the hot partition real bytes, so AQE (and the metrics) see it. We also lower `skewedPartitionThresholdInBytes` because at laptop scale the hot partition is still under AQE's 256 MB default.

In [ ]:
apply_profile(spark, "constrained")
profile_summary(spark)

broken_join = fact.join(dim, on="customer_id", how="inner")
m_broken = measure(spark, "skewed (AQE off, SMJ)", lambda: broken_join.agg(F.sum("amount")).collect())
print("\nbroken-join metrics:", m_broken)

## 2. Detect it — read the Spark UI

http://localhost:4040 -> **SQL / DataFrame** -> click this query -> the join's reduce **Stage** ->
**Tasks -> Summary Metrics**. The tell: **Max task time >> Median**, and one task with a huge
**Shuffle Read** (often **Spill** too). The cell below prints the same thing as a number —
`skew_ratio = max / median`.

In [ ]:
print(f"max task time    : {m_broken['task_time_max_ms']} ms")
print(f"median task time : {m_broken['task_time_median_ms']} ms")
print(f"SKEW RATIO       : {m_broken['skew_ratio']}x   <- one straggler doing ~all the work")

## 3. Fix it (A) — Salting

Spread the hot key across `S` partitions: add a random `salt` to the fact, replicate the
dimension across all salts, and join on `(customer_id, salt)`. Still AQE/broadcast off, so this
isolates the salting effect.

In [ ]:
S = 16
salted_fact = fact.withColumn("salt", F.floor(F.rand(seed=1) * S).cast("int"))
salted_dim  = dim.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(S)])))
salted_join = salted_fact.join(salted_dim, on=["customer_id", "salt"], how="inner")

m_salt = measure(spark, "salted", lambda: salted_join.agg(F.sum("amount")).collect())
print("salted-join metrics:", m_salt)

## 3. Fix it (B) — AQE skew-join

Let Spark split the skewed partition at runtime. We keep broadcast **off** (so it stays a
sort-merge join) and **lower the skew threshold** — our data is tiny, so AQE's 256 MB default
would never trip on a laptop. On real (large) data the defaults work as-is.

In [ ]:
apply_profile(spark, "tuned", **{
    "spark.sql.autoBroadcastJoinThreshold": "-1",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "1m",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "2",
})
profile_summary(spark)

aqe_join = fact.join(dim, on="customer_id", how="inner")
m_aqe = measure(spark, "AQE skew-join", lambda: aqe_join.agg(F.sum("amount")).collect())
print("AQE skew-join metrics:", m_aqe)

## 3. Fix it (C) — Broadcast the small side

If one side fits in memory, broadcast it: no shuffle of the huge fact at all, so skew is moot.
Usually the cheapest fix — reach for it first when a side is small.

In [ ]:
apply_profile(spark, "tuned")   # broadcast enabled (10 MB threshold)
bcast_join = fact.join(F.broadcast(dim), on="customer_id", how="inner")
m_bcast = measure(spark, "broadcast", lambda: bcast_join.agg(F.sum("amount")).collect())
print("broadcast-join metrics:", m_bcast)

## 4. Prove it

Before/after. Watch the **Skew (max / median)** row collapse from tens-of-x toward ~1x, and
runtime / max-task-time / spill fall with it.

In [ ]:
compare([m_broken, m_salt, m_aqe, m_bcast])

## Takeaways & "in real production..."

- **Detect** skew by the task-time **max-vs-median** spread and one fat **Shuffle Read** task —
  averages hide the straggler.
- **Prefer broadcast** when one side fits; else **AQE skew-join** (on by default in Spark 3.2+/4.x);
  fall back to **salting** for explicit control.
- **The "shrink the box" trick:** our data is tiny, so we lowered AQE's skew threshold to
  reproduce the behavior — on real data the defaults trip on their own.
- **In prod:** alert on per-stage task-time skew (max/median), keep AQE enabled, set
  `autoBroadcastJoinThreshold` deliberately, and pre-aggregate / isolate known mega-keys.

## Teardown

Nothing was written (we only counted generated data), so there is nothing to delete. We just
restore the production-tuned safety nets.

In [ ]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")